# ProcessBehavior Tutorial: Fill Weight Analysis

This notebook demonstrates how to use **ProcessBehavior** for analyzing fill weight data from a production process.

## What We'll Cover
1. Loading and exploring the data
2. Understanding your data structure (Sampling Design State)
3. Running automatic analysis
4. Interpreting control charts
5. Detecting signals and special causes
6. Exporting results to Excel

## 1. Setup and Data Loading

First, let's import the library and load our fill weight data.

In [ ]:
# Import required libraries
import pandas as pd
from processbehavior import ProcessDataFrame

# Load the fill weight data
df = pd.read_csv('processbehavior/datasets/data/FILLWEIGHTDATA_800.csv')

# Take a look at the data
print(f"Dataset: {len(df)} observations")
df.head(10)

### Understanding the Data

Our dataset has:
- **pull**: Time sequence (production pull number)
- **lane**: Production lane (1-4)
- **phase**: Phase indicator (1 or 2)
- **fill_weight**: The measurement we want to analyze

In [ ]:
# Check data structure
print("Data Info:")
print(f"  • Time points (pulls): {df['pull'].nunique()}")
print(f"  • Lanes: {sorted(df['lane'].unique())}")
print(f"  • Phases: {sorted(df['phase'].unique())}")
print(f"\nFill Weight Statistics:")
print(df['fill_weight'].describe())

## 2. Create ProcessDataFrame

The `ProcessDataFrame` wraps your data and provides intelligent analysis capabilities.

**Key feature**: It automatically handles missing values and garbage characters!

In [ ]:
# Create ProcessDataFrame - this automatically cleans the data
pdf = ProcessDataFrame(df)

# Check for any data cleaning that happened
print(f"ProcessDataFrame ready with {len(pdf.data)} observations")

## 3. Run Analysis

ProcessBehavior will:
1. **Detect your data structure** (Sampling Design State)
2. **Recommend the best chart type** for your data
3. **Calculate control limits** using Wheeler's methodology
4. **Compute residuals** for variance decomposition

All you need to specify is:
- What to measure (`response_var`)
- What defines groups (`grouping_vars`)
- What defines time (`time_var`)

In [ ]:
# Run the analysis
analysis = pdf.analyze(
    response_var='fill_weight',    # What we're measuring
    grouping_vars=['lane', 'phase'], # What defines subgroups
    time_var='pull'                # Time sequence
)

# Calculate results
result = analysis.calculate()

### Understanding the Analysis Output

The banner above tells you:
- **SDS Type**: What data structure was detected
- **Recommended Charts**: Best chart types for your data
- **Available Capabilities**: What analysis features you can use

In [ ]:
# Explore the results
print("Analysis Summary:")
print(f"  • SDS Type: {result.summary['sds']} - {result.summary['sds_description']}")
print(f"  • Chart Type: {result.summary['analysis_type']}")
print(f"  • Replication: {result.summary['replication_type']}")
print(f"\nAvailable Charts: {list(result.charts.keys())}")

## 4. View Control Chart Data

Let's look at the Xbar chart (subgroup means) to see if our process is in control.

In [ ]:
# Get Xbar chart data
xbar_data = result.charts['Xbar']['data']
xbar_stats = result.charts['Xbar']['statistics']

print("Xbar Chart Statistics:")
print(f"  • Center Line: {xbar_stats['center']:.2f}")
print(f"  • UCL (Upper Control Limit): {xbar_stats['ucl']}")
print(f"  • LCL (Lower Control Limit): {xbar_stats['lcl']}")
print(f"\nXbar Chart Data (first 10 subgroups):")
xbar_data.head(10)

## 5. Create Interactive Visualizations

ProcessBehavior uses Plotly for beautiful, interactive charts.

In [ ]:
# Create interactive control charts
fig = result.plot(
    template='processbehavior',  # Clean, professional theme
    width=1200,
    height=600,
    highlight_signals=True       # Highlight points beyond limits
)

# Display the chart (interactive!)
fig.show()

**Try it!** Hover over points, zoom, pan - the charts are fully interactive.

## 6. Detect Signals (Western Electric Rules)

Let's check for special causes using the Western Electric rules.

In [ ]:
# Detect signals on Xbar chart
signals = result.detect_signals(
    chart='Xbar',
    rules=['rule_1', 'rule_2', 'rule_3', 'rule_4']  # All standard WECO rules
)

print(f"Signal Detection Results:")
print(f"  • Signals Found: {signals.has_signals}")
print(f"  • Total Signal Count: {signals.count}")

if signals.has_signals:
    print(f"\nSignal Details:")
    print(signals.violations)

### Understanding the Rules

- **Rule 1**: Point beyond control limits (3σ)
- **Rule 2**: 2 of 3 consecutive points in Zone A (beyond 2σ)
- **Rule 3**: 4 of 5 consecutive points in Zone B (beyond 1σ)
- **Rule 4**: 8 consecutive points on same side of centerline

## 7. Variance Decomposition (Residuals)

ProcessBehavior calculates Wheeler's residuals (R1-R5) to help you understand sources of variation.

In [ ]:
# Check if residuals are available
if result.has_residuals:
    print("Residuals Available: Yes")
    residuals = result.residuals
    print(f"\nResidual Types: {residuals.columns.tolist()}")
    print(f"\nFirst 10 observations with residuals:")
    # Show original data columns + residuals from dataset
    display_cols = ['pull', 'lane', 'phase', 'fill_weight', 'R1', 'R2', 'R3']
    print(result.dataset[display_cols].head(10))
else:
    print("Residuals: Not available for this SDS type")

### What Each Residual Tells You

- **R1**: Deviation from overall mean (raw residual)
- **R2**: Deviation from time period mean (time effects removed)
- **R3**: Deviation from factor mean (factor effects removed)
- **R4**: Deviation from cell mean (factor + time effects removed)
- **R5**: Pure error (all systematic effects removed)

## 8. Check Main Effects

Are there significant differences between lanes? Between phases?

In [ ]:
# Check for main effects
if result.has_effects:
    print("Main Effects Available: Yes\n")
    
    # Lane effects
    if 'lane' in result.effects:
        print("Lane Main Effects:")
        print(result.effects['lane'])
        print()
    
    # Phase effects
    if 'phase' in result.effects:
        print("Phase Main Effects:")
        print(result.effects['phase'])
else:
    print("Main Effects: Not available for this SDS type")

## 9. Export to Excel

Save everything to Excel for further analysis or sharing with the team.

In [ ]:
# Export comprehensive results to Excel
result.to_excel(
    'fillweight_analysis_results.xlsx',
    include_residuals=True,
    include_effects=True,
    include_interactions=True,
    include_full_dataset=True  # Include all calculated values
)

print("✅ Results exported to: fillweight_analysis_results.xlsx")
print("\nExcel workbook includes:")
print("  • Summary sheet with analysis metadata")
print("  • Xbar chart data")
print("  • Sbar chart data")
print("  • Residuals (with original data columns)")
print("  • Main effects")
print("  • Interactions")
print("  • Full dataset (all calculations)")

## 10. Quick Summary

Let's create a concise summary of the analysis.

In [ ]:
print("=" * 80)
print("FILL WEIGHT ANALYSIS SUMMARY")
print("=" * 80)
print(f"\nData Structure:")
print(f"  • Total Observations: {len(df)}")
print(f"  • Lanes: {sorted(df['lane'].unique())}")
print(f"  • Phases: {sorted(df['phase'].unique())}")
print(f"  • Time Points: {df['pull'].nunique()}")

print(f"\nProcess Behavior:")
print(f"  • SDS Type: {result.summary['sds']}")
print(f"  • Analysis: {result.summary['analysis_type']}")
print(f"  • Center: {xbar_stats['center']:.2f}")

print(f"\nSignal Detection:")
signals_xbar = result.detect_signals(chart='Xbar')
print(f"  • Xbar Signals: {signals_xbar.count}")
print(f"  • Process Status: {'⚠️  OUT OF CONTROL' if signals_xbar.has_signals else '✅ IN CONTROL'}")

print(f"\nCapabilities:")
print(f"  • Residuals: {'✓' if result.has_residuals else '✗'}")
print(f"  • Main Effects: {'✓' if result.has_effects else '✗'}")
print(f"  • Interactions: {'✓' if result.has_interactions else '✗'}")
print("=" * 80)

## Next Steps

### Explore Further
1. **Try different stratifications**: Analyze lanes separately with `stratify='lane'`
2. **Compare phases**: Look at differences between Phase 1 and Phase 2
3. **Investigate signals**: If signals were found, investigate root causes
4. **Check S chart**: Look at variation patterns using the Sbar chart

### Learn More
- ProcessBehavior Documentation: [link to docs]
- Wheeler's Methods: Understanding Statistical Process Control
- Western Electric Rules: Signal detection for special causes

---

**Questions?** This notebook demonstrates the core workflow. ProcessBehavior handles the complexity so you can focus on understanding your process!